imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime


paths

In [0]:
src_bronze_product="/Volumes/retail_catalog/retail_bronze/bronze_product_volume"
target_silver_product="/Volumes/retail_catalog/retail_silver/silver_product_volume"
quarantine_product_path="/Volumes/retail_catalog/retail_quarantine/q_product_volume"
audit_path="/Volumes/retail_catalog/audit/audit_volume/etl_audit/"

load bronze df

In [0]:

df_product_bronze=spark.read\
    .format("delta")\
    .load(src_bronze_product)

incremental df

In [0]:

if DeltaTable.isDeltaTable(spark, target_silver_product):
    bronze_table = DeltaTable.forPath(spark, target_silver_product)
    # Get max data_arrival_timestamp
    max_ts_row = bronze_table.toDF().select(max("data_arrival_timestamp")).collect()[0]
    max_ts = max_ts_row[0]  # None if table is empty
    if max_ts is None:
        print("Bronze table is empty. Will load all records.")
else:
    print("silver table not found. Will load all records.(initial load) ")
    max_ts = None  # first load

# Filter source for incremental load
if max_ts:
    df = df_product_bronze.filter(col("data_arrival_timestamp") > max_ts)
else:
    df = df_product_bronze  # first load, take all records

print(f"Number of records to load: {df.count()}")

In [0]:
if df.isEmpty():
    dbutils.notebook.exit("no new data to load into silver product")

filter data based on schema evolution

In [0]:

rescue_df = df.filter(
    col("_rescued_data").isNotNull() &
    (trim(col("_rescued_data")) != "")
)
df=df.filter(col("_rescued_data").isNull())

quarantine schema changed data

In [0]:
if not rescue_df.isEmpty():
    print("data found in rescued column")
    rescue_df = (
    rescue_df.withColumn("layer", lit("silver"))\
             .withColumn("quarantine_timestamp", current_timestamp())
    )
    rescue_df.write\
        .format("delta")\
        .mode("append")\
        .save(quarantine_product_path)
else:
    pass

In [0]:
if df.isEmpty():
    dbutils.notebook.exit("no new data to load into silver product")

drop duplicate

In [0]:
df=df.dropDuplicates(['product_id','data_arrival_timestamp'])

drop nulls on imp columns

In [0]:
df = df.dropna(subset=["product_id", "product_name","data_arrival_timestamp"],how='any')

handle na 

In [0]:
df=df.fillna({
    "category": "Unknown"
})



invalid checks and custom business logics

In [0]:
valid_categories = ["Technology", "Furniture", "Office Supplies", "Sports & Outdoors", "Unknown"]
df = df.filter(col("category").isin(valid_categories))
df = df.filter(col("product_id").startswith("PROD"))

enrich data

In [0]:
df = df.drop("_rescued_data")

soft delete flag column

In [0]:
df = df.withColumn("is_deleted", lit(False))

In [0]:
display(df)

In [0]:
df = df.select(
    col("product_id").cast("string"),
    col("category").cast("string"),
    col("product_name").cast("string"),
    to_timestamp(
        col("data_arrival_timestamp"),
        "yyyy-MM-dd HH:mm:ss"
    ).alias("data_arrival_timestamp"),
    col("is_deleted").cast("boolean")
)

cast data type for all columns

In [0]:
df.write\
    .format("delta")\
    .mode("append")\
    .partitionBy("category")\
    .save(target_silver_product)

add meta data and etl status to audit table

In [0]:

# Count records
records_count = df.count()

if records_count==0:
    msg=" (Source Empty)"
else:
    msg=""

# max timestamp (only if rows exist)
if records_count > 0:
    max_ts = df.select(max("data_arrival_timestamp")).collect()[0][0]
    max_data_ts_row = max_ts if isinstance(max_ts, datetime) else None
else:
    max_data_ts_row = None

# Use Python datetime for load_time
load_time = datetime.now()

# Define schema explicitly
schema = StructType([
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("load_time", TimestampType(), True),
    StructField("records_loaded", LongType(), True),
    StructField("max_data_timestamp", TimestampType(), True)
])

# Prepare audit data (even if 0 rows)
data = [("silver", f"silver_product{msg}", load_time, records_count, max_data_ts_row)]

# Create DataFrame
df_audit = spark.createDataFrame(data, schema)

# Append to audit table
df_audit.write.format("delta") \
    .mode("append") \
    .save(audit_path)

print(f"Audit log updated successfully. Records loaded: {records_count}")
